# เทรน LSTM ทิศทางเดียวสำหรับเกมภาษามือไทย

อัปโหลดไฟล์นี้ใน [Google Colab](https://colab.research.google.com/) เลือก **Runtime → Change runtime type → GPU** แล้ว **Run all** ค่าเริ่มต้นจะโหลด TSL‑51 และเทรนแยก 3 รอบ เพื่อเปรียบเทียบ LSTM แบบพิกัดพื้นฐาน เพิ่มเรขาคณิต และเพิ่มการเคลื่อนไหว ใช้โมเดล **LSTM 1 ชั้น ทิศทางเดียว + Dropout + Linear** เหมือนกันทุกรอบ ทำนาย 52 คลาสจากคลิปที่ตัดมาแล้วคลิปละหนึ่งท่า

โมเดลเดิมในโปรเจกต์วัดได้ accuracy **99.50%** บน validation ของกลุ่มล่าม และ **53.75%** บน `user_sign` ทั้ง 547 คลิป (macro‑F1 **49.49%**) การทดลองนี้ใช้ 156 คลิปสำหรับเลือกฟีเจอร์และปรับ confidence และประเมินด้วยคลิปที่ระบุ `test` อีก 391 คลิป ทั้งสองส่วนมาจากผู้เซ็นคนเดียวกันซึ่งอยู่นอกชุดเทรน หากมี ZIP ของโมเดลเดิม ตั้ง `BASELINE_BUNDLE` เพื่อวัดทั้งสองโมเดลบน **คลิป test เดียวกัน** [metadata ที่ผู้จัดทำเผยแพร่](https://huggingface.co/datasets/Namonpas/thai-sign-language-tsl51/blob/main/metadata/user_sign_metadata.csv)

ผลลัพธ์อยู่ใน ZIP ท้าย Notebook สำหรับ Google Drive ให้ตั้ง `SAVE_TO_DRIVE = True` TSL‑51 ระบุ CC BY‑NC‑SA 4.0 และข้อกำหนดเพิ่มเติมสำหรับข้อมูลจากบุคคลที่สาม [รายละเอียดชุดข้อมูล](https://huggingface.co/datasets/Namonpas/thai-sign-language-tsl51/blob/main/README.md)

## 1. งานวิจัยและค่าที่นำมาทดลอง

| งานวิจัย | สิ่งที่พบและวิธีใช้กับโปรเจกต์ |
|---|---|
| Jintanachaiwat et al. (2024), §3.1–4 | ใช้ MediaPipe Holistic, 30 เฟรม × 1,662 ค่า, LSTM แบบซ้อน 64/128/64 หน่วยและ Adam สถาปัตยกรรมในงานวิจัยและงานนี้ใช้ LSTM ทิศทางเดียว งานวิจัยทดลอง 5 คลาส ได้ test 100% และ real-time 86% พร้อมรายงานข้อจำกัดจากการเก็บข้อมูลผู้เซ็นคนเดียว เรานำ 30 เฟรมและ Adam มาใช้ ส่วน 128 หน่วยชั้นเดียวกับ dropout 0.25 เป็นค่าที่เลือกสำหรับ TSL‑51 |
| คะรานรัมย์ (2024), Chula ETD 74373 | บทคัดย่อเปรียบเทียบ 4 แบบบน 20 ประโยค งานของเขาได้ผลดีที่สุดด้วย EfficientNetB0 + GRU (accuracy 0.78 ในวิทยานิพนธ์) นำวิธีรายงาน accuracy, precision, recall, F1 รายคลาส ขนาดโมเดล เวลาเทรนและเวลา inference มาปรับใช้ การอ้างอิงส่วนนี้มาจากบทคัดย่อที่เผยแพร่ |
| Chalotonpised et al. (2026), TSL‑ONE‑S, §IV–V | ผู้เซ็น 29 คน 184 glosses ใช้ signer-independent split และแยกตำแหน่งมือเทียบลำตัวจากรูปมือภายในกรอบมือ สำหรับ I3D/SPOTER ใช้ 80 epochs, batch 16, Adam ที่ LR 1e‑4, early stopping และลำดับจุดแบบ fixed-length (ยกตัวอย่าง 64 เฟรม) ค่าเหล่านี้ใช้เป็นทางเลือกในเซลล์ตั้งค่าเมื่อทำการทดลองรอบใหม่ |

แหล่งอ่าน [Springer ฉบับเต็ม](https://link.springer.com/article/10.1007/s44163-024-00113-8) · [Chula ETD](https://digital.car.chula.ac.th/chulaetd/74373/) · [TSL‑ONE‑S บทความฉบับเต็ม](https://www.researchgate.net/publication/401631257_TSL-ONE-S_A_Real-World_Thai_Sign_Language_Dataset_With_Deep_Learning_Benchmarks) · [DOI ของ IEEE](https://doi.org/10.1109/ACCESS.2026.3670970)

**ข้อมูลสำหรับรัน** ใช้ keypoints แบบ CSV ของ TSL‑51 ซึ่งมีลำดับจุดตรงกับโปรแกรมเว็บแคมเดิม TSL‑ONE‑S มี NPY แบบ 2D ที่เลือกจุดหัวคนละชุดกับ TSL‑51 จึงใช้เป็นต้นแบบการ normalize และการแบ่งผู้เซ็นใน Notebook นี้ หากได้ MP4 ที่ได้รับอนุญาต ให้สกัด landmark ตามรูปแบบ TSL‑51 และจับคู่ `sign_id` ก่อนใช้กับ `EXTRA_MANIFEST` [ช่องทางขอ MP4 และโหลด keypoints](https://tsl-one.github.io/)

In [ ]:
from pathlib import Path

WORK_DIR = Path("/content/tsl_lstm")
SAVE_TO_DRIVE = False
DOWNLOAD_TSL51 = "training"     # "all" โหลดทั้ง repository, "local" ใช้ไฟล์เดิม
TSL51_DIR = WORK_DIR / "tsl51"   # เปลี่ยนเป็นโฟลเดอร์ใน Drive เมื่อตั้ง "local"
INCLUDE_WEB_SCRAPED = False      # เปิดเฉพาะเมื่อสิทธิ์อนุญาต ใช้ใน train เท่านั้น
EXTRA_MANIFEST = None            # ตัวอย่าง Path("/content/drive/MyDrive/tsl_extra/manifest.csv")

SEED = 42
FRAMES = 30
VAL_FRACTION = 0.20
EPOCHS = 50
BATCH_SIZE = 64
LEARNING_RATE = 0.001
PATIENCE = 8
HIDDEN_SIZE = 128
DROPOUT = 0.25
MAX_AUG_PER_ORIGINAL = 8         # None ใช้คลิป augment ทั้งหมด
FEATURE_MODES = ("coordinates", "geometry", "geometry_motion")
BASELINE_BUNDLE = None           # ตัวอย่าง Path("/content/drive/MyDrive/tsl51_lstm_bundle.zip")

assert FRAMES >= 2 and EPOCHS > 0 and BATCH_SIZE > 0 and PATIENCE > 0
assert LEARNING_RATE > 0 and HIDDEN_SIZE > 0 and 0 <= DROPOUT < 1
assert MAX_AUG_PER_ORIGINAL is None or MAX_AUG_PER_ORIGINAL >= 0

RUN_NAME = "tsl51_plain_lstm"
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = Path("/content/drive/MyDrive/Thai_Sign_Language_Game") / RUN_NAME
else:
    OUTPUT_DIR = WORK_DIR / RUN_NAME

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("huggingface_hub") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub>=0.34,<2"])

from huggingface_hub import snapshot_download

REPO = "Namonpas/thai-sign-language-tsl51"
REVISION = "6d3dde16db90e5cc10c6fd5e8a36e3be18a68df5"
EXPERT_ZIPS = ["expert_primary_02.zip", "expert_primary_03.zip"]
if INCLUDE_WEB_SCRAPED:
    EXPERT_ZIPS.append("expert_scraped.zip")

assert DOWNLOAD_TSL51 in {"training", "all", "local"}
if DOWNLOAD_TSL51 != "local":
    training_paths = [
        "README.md", "metadata/expert_metadata.csv", "metadata/user_sign_metadata.csv",
        "landmarks/user_sign/*.csv",
        *[f"landmarks/{name}" for name in EXPERT_ZIPS],
    ]
    snapshot_download(
        REPO, repo_type="dataset", revision=REVISION, local_dir=TSL51_DIR,
        allow_patterns=training_paths if DOWNLOAD_TSL51 == "training" else None,
        max_workers=4,
    )

for rel in [
    "metadata/expert_metadata.csv", "metadata/user_sign_metadata.csv",
    *[f"landmarks/{name}" for name in EXPERT_ZIPS],
]:
    if not (TSL51_DIR / rel).is_file():
        raise FileNotFoundError(f"ไม่พบไฟล์ {TSL51_DIR / rel}")
print("TSL-51:", TSL51_DIR, "revision:", REVISION if DOWNLOAD_TSL51 != "local" else "local")

### เพิ่มคลิปของโปรเจกต์ (ถ้ามี)

ตั้ง `EXTRA_MANIFEST` ให้ชี้ไปที่ CSV แบบ UTF‑8 เปิด `SAVE_TO_DRIVE` เพื่อเชื่อม Drive หากใช้ path ใต้ `/content/drive` ตัวอย่างหัวตารางและข้อมูล

```csv
landmark_path,sign_id,sign_clean,signer_id,split
clips/p01_yindee_01.csv,ยินดี_var_1,ยินดี,p01,train
clips/p02_yindee_01.csv,ยินดี_var_1,ยินดี,p02,val
clips/p03_yindee_01.csv,ยินดี_var_1,ยินดี,p03,test
```

ตัวอย่างนี้แสดงรูปแบบคอลัมน์ ให้ใช้หลายคลิปจริงต่อคำและหลายผู้เซ็น กำหนดผู้เซ็นแต่ละคนให้อยู่ split เดียว และอย่านำผู้เซ็นในชุดทดสอบ TSL‑51 มาใส่ในชุดเสริม คำใหม่ทุกคำต้องมีคลิปในทั้งสาม split ถ้าเป็นรูปท่าเดิมของ TSL‑51 ให้ใช้ `sign_id` และ `sign_clean` ที่ปรากฏใน metadata ตรงตัว

ถ้าได้รับ MP4 มา ให้ตัดทีละท่าแล้วสกัด MediaPipe Holistic แบบเดียวกับ TSL‑51 ลง CSV หนึ่งไฟล์ต่อคลิป ใช้ชื่อและความหมายของพิกัดตาม `COORDINATE_COLUMNS` ด้านล่าง ใช้ค่าเว้นว่างหรือ NaN เมื่อจับจุดไม่ได้ และบันทึก `frame` กับ `t_ms` หากมี ชุด NPY จาก TSL‑ONE‑S ต้องมีการแปลงจุดและจับคู่ gloss ที่ตรวจสอบแล้วก่อนใช้ [สเปก landmark ของ TSL‑51](https://huggingface.co/datasets/Namonpas/thai-sign-language-tsl51/blob/main/README.md#landmark-format)

In [ ]:
%%writefile /content/tsl_preprocess.py
import csv

import numpy as np


ARM_POINTS = ("l_shoulder", "r_shoulder", "l_elbow", "r_elbow", "l_wrist", "r_wrist")
FACE_POINTS = ("lbrow_outer", "lbrow_inner", "rbrow_inner", "rbrow_outer", "mouth_right", "mouth_left")
COORDINATE_COLUMNS = (
    [f"{side}_{axis}{point}" for side in ("lh", "rh") for point in range(21) for axis in "xyz"]
    + [f"{point}_{axis}" for point in ARM_POINTS + FACE_POINTS for axis in "xyz"]
)
N_POINTS = len(COORDINATE_COLUMNS) // 3
INPUT_FEATURES = 4 * N_POINTS


def landmarks_to_tensor(coordinates, frames, time_ms=None):
    """Signed interval -> float32 array [frames, 216] in the trained feature order."""
    points = np.asarray(coordinates, dtype=np.float32).reshape(-1, N_POINTS, 3)
    if len(points) < 2:
        raise ValueError("Need at least two frames")
    if frames < 2:
        raise ValueError("frames must be at least two")

    valid = np.isfinite(points).all(axis=-1)
    if not valid.any():
        raise ValueError("No detected landmarks")
    points = np.where(valid[..., None], points, 0.0)

    left, right = points[:, 42, :2], points[:, 43, :2]
    widths = np.linalg.norm(left - right, axis=-1)
    anchors = valid[:, 42] & valid[:, 43] & (widths > 0.025)
    if anchors.any():
        indexes = np.arange(len(points))
        center = (left + right) / 2.0
        center = np.column_stack([
            np.interp(indexes, indexes[anchors], center[anchors, axis]) for axis in range(2)
        ])
        points[:, :, :2] -= center[:, None, :]
        points /= np.median(widths[anchors])
    else:
        points[:, :, :2] -= 0.5

    clock = np.arange(len(points), dtype=np.float64)
    if time_ms is not None:
        stamps = np.asarray(time_ms, dtype=np.float64)
        if stamps.shape == clock.shape and np.isfinite(stamps).all() and (np.diff(stamps) > 0).all():
            clock = stamps

    target = np.linspace(clock[0], clock[-1], frames)
    hi = np.searchsorted(clock, target, side="right").clip(1, len(points) - 1)
    lo = hi - 1
    alpha = ((target - clock[lo]) / (clock[hi] - clock[lo]))[:, None, None]
    nearest = np.where(alpha[:, 0, 0] < 0.5, lo, hi)
    both = valid[lo] & valid[hi]
    interpolated = points[lo] + alpha * (points[hi] - points[lo])
    selected = np.where(both[..., None], interpolated, points[nearest])
    selected = np.where(valid[nearest, :, None], selected, 0.0)
    return np.concatenate(
        [selected.reshape(frames, -1), valid[nearest]], axis=-1,
    ).astype(np.float32)


def read_landmark_csv(file, frames):
    """Read an open UTF-8/UTF-8-SIG text stream in the TSL-51 CSV format."""
    reader = csv.reader(file)
    header = next(reader, [])
    index = {name.strip(): i for i, name in enumerate(header)}
    missing = [column for column in COORDINATE_COLUMNS if column not in index]
    if missing:
        raise ValueError(f"Missing landmark columns: {missing[:8]} ({len(missing)} total)")
    columns = [index[name] for name in COORDINATE_COLUMNS]
    time_column = index.get("t_ms")
    values, stamps = [], []
    for row in reader:
        if not row:
            continue
        if len(row) != len(header):
            raise ValueError(f"Wrong column count at CSV line {reader.line_num}")
        values.append([row[i].strip() or "nan" for i in columns])
        if time_column is not None:
            stamps.append(row[time_column].strip() or "nan")
    return landmarks_to_tensor(values, frames, stamps if time_column is not None else None)

## 2. แบ่งข้อมูลและสร้าง cache

- **Train / expert validation** แบ่งตามรหัสคลิปต้นฉบับแบบแยกคลาส โดยชุด validation ใช้คลิปต้นฉบับเท่านั้น เก็บ augmentation จากกลุ่ม train สูงสุด 8 ตัวอย่างต่อคลิปต้นฉบับเพื่อจำกัดน้ำหนักของคลิปซ้ำ
- **Calibration** ใช้คลิป `user_sign` ที่ผู้จัดทำระบุ `calib` เลือกวิธีสร้างฟีเจอร์ ปรับอุณหภูมิของ softmax และเลือกระดับ confidence ขั้นต่ำ
- **Test** ใช้ `user_sign/test` หลังจากเลือกค่าทั้งหมดแล้ว กรณีมีคลิปโปรเจกต์ที่แยกผู้เซ็นออกต่างหากจะมีรายงานเพิ่ม ควรดูผลสองแหล่งตามจำนวนผู้เซ็นและจำนวนคลิปที่รายงาน

In [ ]:
import csv
import hashlib
import importlib
import io
import json
import math
import os
import posixpath
import shutil
import unicodedata
from collections import Counter, defaultdict
from contextlib import ExitStack
from pathlib import PurePosixPath
from zipfile import ZipFile

import numpy as np
from tqdm.auto import tqdm

if "/content" not in sys.path:
    sys.path.insert(0, "/content")
import tsl_preprocess as preprocess
preprocess = importlib.reload(preprocess)


def read_metadata(path, required):
    with path.open(encoding="utf-8-sig", newline="") as stream:
        reader = csv.DictReader(stream)
        missing = set(required) - set(reader.fieldnames or ())
        if missing:
            raise ValueError(f"{path}: missing columns {sorted(missing)}")
        return [{k: (row[k] or "").strip() for k in required} for row in reader]


def make_splits(root, include_web, val_fraction, seed, extra_manifest=None, max_aug=None):
    if not 0 < val_fraction < 1:
        raise ValueError("VAL_FRACTION must be between 0 and 1")

    common = ("video_id", "source", "sign_id", "sign_clean", "landmark_path", "extraction_status")
    expert = read_metadata(root / "metadata/expert_metadata.csv", (
        *common, "source_group", "original_video_id", "is_augmented",
    ))
    user = read_metadata(root / "metadata/user_sign_metadata.csv", (*common, "subset"))
    if any(r["extraction_status"] != "ok" for r in expert + user):
        print("Metadata rows with extraction errors will be excluded.")
    expert = [r for r in expert if r["extraction_status"] == "ok"]
    user = [r for r in user if r["extraction_status"] == "ok"]

    original_by_class = defaultdict(list)
    group_by_id = {}
    for row in expert:
        if row["source_group"] not in {"expert-primary", "web-scraped"}:
            raise ValueError(f"Unknown expert source group: {row['source_group']}")
        if row["source_group"] == "web-scraped" and not include_web:
            continue
        if row["is_augmented"].lower() not in {"true", "false"} or not row["original_video_id"]:
            raise ValueError(f"Invalid augmentation metadata: {row['video_id']}")
        group = (row["source"], row["original_video_id"])
        group_by_id.setdefault(group, []).append(row)

    for group, variants in group_by_id.items():
        originals = [r for r in variants if r["is_augmented"].lower() == "false"]
        if len(originals) != 1 or len({r["sign_id"] for r in variants}) != 1:
            raise ValueError(f"Invalid original-clip group: {group}")
        if originals[0]["source_group"] == "expert-primary":
            original_by_class[originals[0]["sign_id"]].append(group)

    if len(original_by_class) != 52 or "null_act" not in original_by_class:
        raise ValueError(f"Expected 52 TSL-51 primary classes: got {len(original_by_class)}")
    if {r["sign_id"] for r in user} != set(original_by_class):
        raise ValueError("TSL-51 user metadata has unexpected or missing classes")

    rng = np.random.default_rng(seed)
    val_groups = set()
    for label, groups in sorted(original_by_class.items()):
        groups.sort()
        if len(groups) < 2:
            raise ValueError(f"{label}: need two or more originals")
        count = min(len(groups) - 1, max(1, round(len(groups) * val_fraction)))
        val_groups.update(groups[i] for i in rng.permutation(len(groups))[:count])

    rows = []
    for group, variants in sorted(group_by_id.items()):
        original = [r for r in variants if r["is_augmented"].lower() == "false"]
        variants = sorted((r for r in variants if r["is_augmented"].lower() == "true"),
                          key=lambda r: r["video_id"])
        if group in val_groups:
            variants = []
        elif max_aug is not None and len(variants) > max_aug:
            chosen = rng.choice(len(variants), size=max_aug, replace=False)
            variants = [variants[i] for i in sorted(chosen)]
        for row in original + variants:
            rows.append({
                **row, "storage": "expert", "split": "val" if group in val_groups else "train",
                "group": "/".join(group),
            })
    for row in user:
        if row["subset"] not in {"calib", "test"}:
            raise ValueError(f"Unexpected user_sign subset: {row['subset']}")
        rows.append({**row, "storage": "user", "split": row["subset"], "group": row["source"]})
    if len({r["video_id"] for r in user}) != len(user) or len({r["landmark_path"] for r in user}) != len(user):
        raise ValueError("Duplicate user_sign clip IDs or paths")

    if extra_manifest is not None:
        extra_manifest = Path(extra_manifest)
        extra = read_metadata(extra_manifest, ("landmark_path", "sign_id", "sign_clean", "signer_id", "split"))
        if not extra:
            raise ValueError("EXTRA_MANIFEST has no clips")
        split_by_signer = {}
        used_paths = set()
        for row in extra:
            signer, split = row["signer_id"], row["split"].lower()
            if not all(row.values()) or split not in {"train", "val", "test"}:
                raise ValueError(f"Invalid extra-manifest row: {row}")
            if split_by_signer.setdefault(signer, split) != split:
                raise ValueError(f"Signer {signer} appears in more than one split")
            path = (extra_manifest.parent / row["landmark_path"]).resolve(strict=True)
            if path in used_paths or path.is_relative_to(root.resolve()):
                raise ValueError(f"Invalid or repeated extra clip: {path}")
            used_paths.add(path)
            rows.append({
                **row, "landmark_path": str(path), "storage": "extra", "split": split,
                "source": "project_extra", "group": signer, "video_id": path.stem,
            })

    names = {}
    for row in rows:
        label = row["sign_id"]
        name = "ไม่พบรูปท่า" if label == "null_act" else row["sign_clean"]
        if not label or not name or names.setdefault(label, name) != name:
            raise ValueError(f"Inconsistent Thai name for {label}: {name}")
    return rows, names

In [ ]:
def zip_key(path):
    path = posixpath.normpath(path.replace("\\", "/"))
    return unicodedata.normalize("NFC", PurePosixPath(path).name)


def file_stamp(path):
    stat = path.stat()
    return [str(path), stat.st_size, stat.st_mtime_ns]


def build_feature_cache(rows, root, archives, frames, cache_dir):
    # Metadata paths use earlier folder names for the primary ZIPs.
    # Match the unique normalized CSV basename across archives.
    files = [root / "metadata" / name for name in ("expert_metadata.csv", "user_sign_metadata.csv")]
    files.extend(root / "landmarks" / archive for archive in archives)
    files.extend(root / r["landmark_path"] for r in rows if r["storage"] == "user")
    files.extend(Path(r["landmark_path"]) for r in rows if r["storage"] == "extra")
    fingerprint = hashlib.sha256(json.dumps([
        hashlib.sha256(Path(preprocess.__file__).read_bytes()).hexdigest(),
        frames, [file_stamp(path) for path in files], rows,
    ], ensure_ascii=False, sort_keys=True).encode("utf-8")).hexdigest()[:20]

    cache_dir.mkdir(exist_ok=True, parents=True)
    array_path = cache_dir / f"{fingerprint}.npy"
    audit_path = cache_dir / f"{fingerprint}_audit.json"
    if array_path.is_file() and audit_path.is_file():
        return array_path, json.loads(audit_path.read_text(encoding="utf-8"))

    shape = (len(rows), frames, preprocess.INPUT_FEATURES)
    required = math.prod(shape) * np.dtype(np.float32).itemsize
    if shutil.disk_usage(cache_dir).free < required + 100 * 1024**2:
        raise OSError(f"Feature cache needs {required / 1024**3:.2f} GiB of free space")

    temp_path = array_path.with_suffix(".partial")
    with ExitStack() as stack:
        index = defaultdict(list)
        for archive in archives:
            opened = stack.enter_context(ZipFile(root / "landmarks" / archive))
            for member in opened.infolist():
                if not member.is_dir() and member.filename.lower().endswith(".csv"):
                    index[zip_key(member.filename)].append((opened, member))
        sources = []
        for row in rows:
            path = row["landmark_path"]
            if row["storage"] == "expert":
                matches = index.get(zip_key(path), [])
                if len(matches) != 1:
                    raise ValueError(f"Expected one ZIP member for {path}; found {len(matches)}")
                sources.append(matches[0])
            else:
                path = root / path if row["storage"] == "user" else Path(path)
                if not path.is_file():
                    raise FileNotFoundError(path)
                sources.append(path)

        features = np.lib.format.open_memmap(temp_path, mode="w+", dtype=np.float32, shape=shape)
        rejected = []
        try:
            for i, source in enumerate(tqdm(sources, desc="แปลง landmark")):
                if isinstance(source, tuple):
                    archive, member = source
                    stream = io.TextIOWrapper(archive.open(member), encoding="utf-8-sig", newline="")
                else:
                    stream = source.open(encoding="utf-8-sig", newline="")
                with stream:
                    try:
                        features[i] = preprocess.read_landmark_csv(stream, frames)
                    except ValueError as exc:
                        reason = str(exc)
                        if reason not in {"Need at least two frames", "No detected landmarks"}:
                            raise ValueError(f"{rows[i]['landmark_path']}: {reason}") from exc
                        rejected.append({"index": i, "reason": reason})
            features.flush()
        finally:
            del features

    os.replace(temp_path, array_path)
    temporary_audit = audit_path.with_suffix(".partial")
    temporary_audit.write_text(json.dumps(rejected, ensure_ascii=False), encoding="utf-8")
    os.replace(temporary_audit, audit_path)
    return array_path, rejected


rows, thai_names = make_splits(TSL51_DIR, INCLUDE_WEB_SCRAPED, VAL_FRACTION, SEED, EXTRA_MANIFEST, MAX_AUG_PER_ORIGINAL)
feature_path, rejected = build_feature_cache(rows, TSL51_DIR, EXPERT_ZIPS, FRAMES, WORK_DIR / "cache")
rejected_by_index = {r["index"]: r["reason"] for r in rejected}
features = np.load(feature_path, mmap_mode="c")
assert features.shape == (len(rows), FRAMES, preprocess.INPUT_FEATURES)

labels = sorted(thai_names)
label_index = {label: i for i, label in enumerate(labels)}
targets = np.array([label_index[r["sign_id"]] for r in rows], dtype=np.int64)
split_indices = {
    split: np.array([
        i for i, r in enumerate(rows) if r["split"] == split and i not in rejected_by_index
    ], dtype=np.int64)
    for split in ("train", "val", "calib", "test")
}
for split, indices in split_indices.items():
    expected = set(labels) if split != "calib" else {r["sign_id"] for r in rows if r["storage"] == "user"}
    missing_labels = expected - {rows[i]["sign_id"] for i in indices}
    if missing_labels:
        raise ValueError(f"{split} is missing valid clips for: {sorted(missing_labels)}")

train_groups = {rows[i]["group"] for i in split_indices["train"] if rows[i]["storage"] == "expert"}
val_groups = {rows[i]["group"] for i in split_indices["val"] if rows[i]["storage"] == "expert"}
assert train_groups.isdisjoint(val_groups)
assert all(rows[i].get("is_augmented", "").lower() != "true"
           for split in ("val", "calib", "test") for i in split_indices[split])

with (OUTPUT_DIR / "splits.csv").open("w", encoding="utf-8-sig", newline="") as stream:
    columns = ["split", "source", "video_id", "sign_id", "group", "rejected_reason"]
    writer = csv.DictWriter(stream, fieldnames=columns)
    writer.writeheader()
    for i, row in enumerate(rows):
        writer.writerow({**{col: row[col] for col in columns[:-1]},
                         "rejected_reason": rejected_by_index.get(i, "")})

print(f"คลาส {len(labels)} | คลิปที่ตัดออก {len(rejected)}")
for split, indices in split_indices.items():
    print(split, len(indices), dict(Counter(rows[i]["source"] for i in indices)))

## 3. Feature engineering และ LSTM

| โหมด | ฟีเจอร์ที่ส่งเข้า LSTM |
|---|---|
| `coordinates` | 216 ค่า ได้แก่ XYZ ของ 54 จุด และ mask 54 ค่า พิกัด XY อ้างอิงจุดกึ่งกลางไหล่ หารด้วยความกว้างไหล่มัธยฐานของคลิป |
| `geometry` | พิกัดพื้นฐาน + รูปมือเทียบข้อมือและขนาดฝ่ามือ + cosine ของมุมข้อนิ้ว/ข้อศอก + ระยะ XY ระหว่างจุดสำคัญ เช่น ปลายนิ้วชี้สองข้าง |
| `geometry_motion` | เรขาคณิต + การเปลี่ยนตำแหน่ง XY ของข้อมือและปลายนิ้วตามเวลา พร้อม mask ของคู่เฟรมที่ตรวจจับได้ |

ให้ $p_{t,i}$ เป็นจุดที่ normalize แล้ว สูตรที่เพิ่มคือ

- **รูปมือ** $r_{t,i} = (p_{t,i}-p_{t,\mathrm{wrist}})/\lVert(p_{t,\mathrm{MCP9}}-p_{t,\mathrm{wrist}})_{xy}\rVert_2$
- **มุม** $\cos\theta=(u\cdot v)/(\lVert u\rVert_2\lVert v\rVert_2)$ สำหรับเวกเตอร์กระดูกสองด้านของข้อ ใช้ 3D ในมือเดียวกัน และ 2D สำหรับข้อศอก
- **ระยะ** $d_{ij}=\lVert (p_{t,i}-p_{t,j})_{xy}\rVert_2$ การใช้เฉพาะ XY สำหรับระยะระหว่างมือ ใบหน้า และลำตัวช่วยเลี่ยงการเทียบ Z ที่อ้างอิงคนละจุด
- **การเคลื่อนไหว** $v_{t,i}=(p_{t,i}-p_{t-1,i})_{xy}(T-1)$ หลัง resample เป็น $T$ เฟรม เป็นอัตราเปลี่ยนต่อสัดส่วนระยะเวลาของคลิป มีการจำกัดค่าสุดโต่งและใช้ศูนย์เมื่อไม่พบจุดในคู่เฟรม

โหมดทั้งสามใช้ข้อมูล สถาปัตยกรรม และค่าฝึกชุดเดียวกัน จำนวน weight ใน LSTM เปลี่ยนตามจำนวนฟีเจอร์ รายงาน `ablation.csv` แสดงผลของแต่ละโหมด ฟีเจอร์เรขาคณิตเป็นตัวแปรที่ให้โมเดลเรียนรู้จากข้อมูล ไม่ใช่คะแนนคุณภาพการทำท่า

ที่มาของแนวคิด [TSL‑ONE‑S §IV‑C](https://doi.org/10.1109/ACCESS.2026.3670970) · [Roh et al., 2024](https://www.sign-lang.uni-hamburg.de/lrec/pub/24052.html) · [Belissen et al., §2.3.1](https://www.mdpi.com/1999-4893/13/12/310)

In [ ]:
import random
import time

import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training device:", device)
VALID_MODES = ("coordinates", "geometry", "geometry_motion")
assert FEATURE_MODES and len(set(FEATURE_MODES)) == len(FEATURE_MODES)
assert set(FEATURE_MODES) <= set(VALID_MODES)


def joint_cosines(points, mask, triples):
    a, b, c = triples.unbind(dim=1)
    first = points.index_select(-2, a) - points.index_select(-2, b)
    second = points.index_select(-2, c) - points.index_select(-2, b)
    norm = torch.linalg.vector_norm(first, dim=-1) * torch.linalg.vector_norm(second, dim=-1)
    valid = mask.index_select(-1, a) & mask.index_select(-1, b) & mask.index_select(-1, c)
    value = (first.mul(second).sum(dim=-1) / norm.clamp_min(1e-6)).clamp(-1, 1)
    return torch.where(valid & (norm > 1e-6), value, 0.0)


class SignLSTM(nn.Module):
    def __init__(self, mode, classes, hidden, dropout):
        super().__init__()
        self.mode = mode
        fingers = [(base, base + 1, base + 2) for base in (1, 5, 9, 13, 17)]
        fingers += [(a + 1, b + 1, c + 1) for a, b, c in fingers]
        self.register_buffer("fingers", torch.tensor(fingers))
        self.register_buffer("elbows", torch.tensor([(42, 44, 46), (43, 45, 47)]))
        self.register_buffer("pairs", torch.tensor([
            (0, 21), (8, 29), (4, 8), (25, 29), (8, 52), (29, 53), (0, 44), (21, 45),
        ]))
        self.register_buffer("motion_ids", torch.tensor([0, 4, 8, 12, 16, 20, 21, 25, 29, 33, 37, 41]))
        geometry_features = 126 + 2 * len(self.fingers) + len(self.elbows) + len(self.pairs)
        motion_features = 3 * len(self.motion_ids)
        width = preprocess.INPUT_FEATURES
        if mode != "coordinates":
            width += geometry_features
        if mode == "geometry_motion":
            width += motion_features
        self.lstm = nn.LSTM(width, hidden, num_layers=1, batch_first=True, bidirectional=False)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden, classes)

    def extract_features(self, x):
        if self.mode == "coordinates":
            return x

        points = x[..., :162].unflatten(-1, (54, 3))
        mask = x[..., 162:] > 0.5
        xy = points[..., :2]
        hands = points[..., :42, :].unflatten(-2, (2, 21))
        hand_mask = mask[..., :42].unflatten(-1, (2, 21))
        local = hands - hands[..., :1, :]
        palm = torch.linalg.vector_norm(local[..., 9, :2], dim=-1)
        palm_valid = hand_mask[..., 0] & hand_mask[..., 9] & (palm > 0.025)
        local = (local / palm.clamp_min(0.025)[..., None, None]).clamp(-4, 4)
        local = torch.where((hand_mask & palm_valid[..., None])[..., None], local, 0.0)
        finger_angles = joint_cosines(hands, hand_mask, self.fingers).flatten(-2)
        elbow_angles = joint_cosines(xy, mask, self.elbows)

        a, b = self.pairs.unbind(dim=1)
        distance = torch.linalg.vector_norm(xy.index_select(-2, a) - xy.index_select(-2, b), dim=-1)
        pair_valid = mask.index_select(-1, a) & mask.index_select(-1, b)
        distance = torch.where(pair_valid, distance, 0.0)
        engineered = [x, local.flatten(-3), finger_angles, elbow_angles, distance]

        if self.mode == "geometry_motion":
            trajectory = xy.index_select(-2, self.motion_ids)
            found = mask.index_select(-1, self.motion_ids)
            step_valid = found[:, 1:] & found[:, :-1]
            delta = (trajectory[:, 1:] - trajectory[:, :-1]) * (x.size(1) - 1)
            delta = torch.where(step_valid[..., None], delta.clamp(-8, 8), 0.0)
            velocity = torch.cat([torch.zeros_like(trajectory[:, :1]), delta], dim=1)
            velocity_mask = torch.cat([torch.zeros_like(found[:, :1]), step_valid], dim=1)
            engineered.extend([velocity.flatten(-2), velocity_mask.to(x.dtype)])
        return torch.cat(engineered, dim=-1)

    def forward(self, x):
        _, (hidden, _) = self.lstm(self.extract_features(x))
        return self.classifier(self.dropout(hidden[-1]))


class ClipDataset(Dataset):
    def __init__(self, data, targets, indices):
        self.data, self.targets, self.indices = data, targets, indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        i = self.indices[index]
        return torch.from_numpy(self.data[i]), self.targets[i]


def confusion(truth, predicted, classes):
    return np.bincount(classes * truth + predicted, minlength=classes**2).reshape(classes, classes)


def class_scores(matrix):
    correct = matrix.diagonal()
    support, predicted = matrix.sum(axis=1), matrix.sum(axis=0)
    return (correct / np.maximum(predicted, 1), correct / np.maximum(support, 1),
            2 * correct / np.maximum(support + predicted, 1), support)


def metrics(truth, predicted):
    matrix = confusion(truth, predicted, len(labels))
    precision, recall, f1, support = class_scores(matrix)
    present = support > 0
    return {
        "accuracy": float(matrix.trace() / matrix.sum()),
        "macro_precision": float(precision[present].mean()),
        "macro_recall": float(recall[present].mean()),
        "macro_f1": float(f1[present].mean()),
        "clips": int(matrix.sum()),
        "evaluated_classes": int(present.sum()),
    }


@torch.inference_mode()
def collect_logits(model, loader):
    model.eval()
    batches, truth = [], []
    for x, y in loader:
        batches.append(model(x.to(device, non_blocking=True)).cpu().numpy())
        truth.append(y.numpy())
    return np.concatenate(truth), np.concatenate(batches)


def loader_for(data, y, indices, shuffle=False):
    return DataLoader(
        ClipDataset(data, y, indices), batch_size=BATCH_SIZE,
        shuffle=shuffle, pin_memory=device.type == "cuda",
        generator=torch.Generator().manual_seed(SEED),
    )


def write_table(path, records):
    with path.open("w", encoding="utf-8-sig", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=records[0].keys())
        writer.writeheader()
        writer.writerows(records)


for mode in FEATURE_MODES:
    trial = SignLSTM(mode, len(labels), HIDDEN_SIZE, DROPOUT)
    probe = torch.zeros(2, FRAMES, preprocess.INPUT_FEATURES)
    assert torch.isfinite(trial.extract_features(probe)).all()
    assert trial(probe).shape == (2, len(labels))
    assert not trial.lstm.bidirectional and trial.lstm.num_layers == 1

In [ ]:
counts = np.bincount(targets[split_indices["train"]], minlength=len(labels))
weights = 1 / np.sqrt(counts)
loss_function = nn.CrossEntropyLoss(weight=torch.tensor(weights / weights.mean(), dtype=torch.float32, device=device))
val_loader = loader_for(features, targets, split_indices["val"])
calib_loader = loader_for(features, targets, split_indices["calib"])


def train_mode(mode):
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    model = SignLSTM(mode, len(labels), HIDDEN_SIZE, DROPOUT).to(device)
    train_loader = loader_for(features, targets, split_indices["train"], shuffle=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", patience=3, factor=0.5)
    history = []
    best_score, best_epoch = -1.0, 0
    checkpoint = OUTPUT_DIR / f"{mode}_weights.pt"
    start = time.perf_counter()
    for epoch in range(1, EPOCHS + 1):
        model.train()
        loss_sum, weight_sum = torch.zeros((), device=device), torch.zeros((), device=device)
        hits = torch.zeros((), device=device)
        rate = optimizer.param_groups[0]["lr"]
        for x, y in tqdm(train_loader, desc=f"{mode} {epoch}/{EPOCHS}", leave=False):
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = loss_function(logits, y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            batch_weight = loss_function.weight[y].sum()
            loss_sum += loss.detach() * batch_weight
            weight_sum += batch_weight
            hits += (logits.detach().argmax(dim=1) == y).sum()

        train_loss = (loss_sum / weight_sum).item()
        if not math.isfinite(train_loss):
            raise RuntimeError(f"{mode}: training loss is not finite")
        y_val, logits_val = collect_logits(model, val_loader)
        val = metrics(y_val, logits_val.argmax(axis=1))
        scheduler.step(val["macro_f1"])
        history.append({
            "epoch": epoch, "train_loss": train_loss, "train_accuracy": hits.item() / len(train_loader.dataset),
            "val_accuracy": val["accuracy"], "val_macro_f1": val["macro_f1"], "learning_rate": rate,
        })
        write_table(OUTPUT_DIR / f"{mode}_history.csv", history)
        print(f"{mode} {epoch:02d} | loss {train_loss:.4f} | expert val macro-F1 {val['macro_f1']:.4f}")
        if val["macro_f1"] > best_score:
            best_score, best_epoch, best_val_acc = val["macro_f1"], epoch, val["accuracy"]
            temporary = checkpoint.with_suffix(".partial")
            torch.save(model.state_dict(), temporary)
            os.replace(temporary, checkpoint)
        elif epoch - best_epoch >= PATIENCE:
            break

    elapsed = time.perf_counter() - start
    model.load_state_dict(torch.load(checkpoint, map_location=device, weights_only=True))
    y_calib, logits_calib = collect_logits(model, calib_loader)
    calib = metrics(y_calib, logits_calib.argmax(axis=1))
    return {
        "mode": mode, "lstm_input_features": model.lstm.input_size,
        "parameters": sum(p.numel() for p in model.parameters()),
        "best_epoch": best_epoch, "expert_val_accuracy": best_val_acc,
        "expert_val_macro_f1": best_score, "user_calib_accuracy": calib["accuracy"],
        "user_calib_macro_f1": calib["macro_f1"], "train_seconds": elapsed,
    }


ablation = [train_mode(mode) for mode in FEATURE_MODES]
write_table(OUTPUT_DIR / "ablation.csv", ablation)
winner = max(ablation, key=lambda row: (row["user_calib_macro_f1"], row["user_calib_accuracy"], -row["parameters"]))
print(json.dumps(ablation, indent=2))
selected_mode = winner["mode"]
checkpoint = OUTPUT_DIR / f"{selected_mode}_weights.pt"
model = SignLSTM(selected_mode, len(labels), HIDDEN_SIZE, DROPOUT).to(device)
model.load_state_dict(torch.load(checkpoint, map_location=device, weights_only=True))
print("Selected from user_calib:", selected_mode)

## 4. ความถูกต้องของคำและความน่าเชื่อถือของ confidence

- **Accuracy** = จำนวนคลิปที่ทายถูก ÷ จำนวนคลิปทั้งหมด **Precision** และ **recall** วัดรายคำจาก $TP/(TP+FP)$ และ $TP/(TP+FN)$ ตามลำดับ ส่วน $F_1=2PR/(P+R)$ และ **macro-F1** คือค่าเฉลี่ยของคลาสที่มีตัวอย่างจริง ดูคู่ที่ทายสลับกันใน confusion CSV
- **Temperature scaling** หา $T>0$ จากคลิป `calib` เพื่อลด negative log-likelihood ของ $p=\mathrm{softmax}(z/T)$ การปรับ $T$ มีผลต่อ confidence แต่ลำดับคำทำนายเดิมยังคงเดิม
- **ECE** = $\sum_b n_b/N \cdot |\mathrm{accuracy}_b-\mathrm{confidence}_b|$ จัด confidence เป็น 10 ช่วง **Brier** = $\frac1N\sum_i\sum_k(p_{ik}-\mathbb{1}[y_i=k])^2$ ค่า ECE, Brier และ NLL ยิ่งต่ำยิ่งดีเมื่อวัดบนชุดเดียวกัน
- **เกณฑ์ปฏิเสธคำทาย** เลือกจาก `calib` ตาม macro-F1 เมื่อ confidence ต่ำกว่าเกณฑ์จะแสดง `null_act` รายงาน coverage ของหน้าจอและอัตราทายเป็นคำในคลิปพัก เพื่อดูผลของเกณฑ์นี้

อ่านผล `metrics.json` ควบคู่ `*_per_class.csv` และ `*_predictions.csv` ค่า confidence บอกความมั่นใจของคำที่ทาย ส่วนการให้คะแนนว่าผู้เล่นทำท่าดีแค่ไหนต้องมีตัวอย่างอ้างอิงและป้ายกำกับคุณภาพท่า [Guo et al., 2017](https://proceedings.mlr.press/v70/guo17a.html)

In [ ]:
def probabilities(logits, temperature=1.0):
    scaled = logits.astype(np.float64) / temperature
    shifted = scaled - scaled.max(axis=1, keepdims=True)
    exp = np.exp(shifted)
    return exp / exp.sum(axis=1, keepdims=True)


def log_loss(truth, scores):
    return float(-np.log(scores[np.arange(len(truth)), truth].clip(1e-15, 1)).mean())


def confidence_quality(truth, scores, name):
    confidence = scores.max(axis=1)
    hits = scores.argmax(axis=1) == truth
    bins = np.minimum((confidence * 10).astype(int), 9)
    records, ece = [], 0.0
    for i in np.unique(bins):
        mask = bins == i
        count = int(mask.sum())
        acc, mean_conf = float(hits[mask].mean()), float(confidence[mask].mean())
        records.append({"score": name, "lower": i / 10, "upper": (i + 1) / 10,
                        "samples": count, "accuracy": acc, "confidence": mean_conf})
        ece += count / len(truth) * abs(acc - mean_conf)
    one_hot = np.eye(scores.shape[1])[truth]
    return {"ece": ece, "nll": log_loss(truth, scores),
            "brier": float(np.square(scores - one_hot).sum(axis=1).mean())}, records


null_id = label_index["null_act"]
def decide(scores, threshold):
    return np.where(scores.max(axis=1) >= threshold, scores.argmax(axis=1), null_id)


y_calib, logits_calib = collect_logits(model, calib_loader)
temperatures = [1.0, *np.geomspace(0.25, 10.0, 121)]
temperature = float(min(temperatures, key=lambda t: log_loss(y_calib, probabilities(logits_calib, t))))
calib_scores = probabilities(logits_calib, temperature)
threshold = float(max(np.linspace(0.0, 0.95, 96), key=lambda t: metrics(
    y_calib, decide(calib_scores, t),
)["macro_f1"]))
assert np.array_equal(calib_scores.argmax(axis=1), logits_calib.argmax(axis=1))


def evaluate_test(name, indices):
    truth, logits = collect_logits(model, loader_for(features, targets, indices))
    raw = probabilities(logits)
    scores = probabilities(logits, temperature)
    raw_pred, decision = logits.argmax(axis=1), decide(scores, threshold)
    matrix = confusion(truth, decision, len(labels))
    precision, recall, f1, support = class_scores(matrix)
    write_table(OUTPUT_DIR / f"{name}_per_class.csv", [
        {"sign_id": label, "thai": thai_names[label], "support": support[i],
         "precision": precision[i], "recall": recall[i], "f1": f1[i]}
        for i, label in enumerate(labels)
    ])
    with (OUTPUT_DIR / f"{name}_confusion.csv").open("w", encoding="utf-8-sig", newline="") as stream:
        writer = csv.writer(stream)
        writer.writerow(["actual/predicted", *labels])
        writer.writerows([label, *counts] for label, counts in zip(labels, matrix))
    write_table(OUTPUT_DIR / f"{name}_predictions.csv", [
        {"video_id": rows[i]["video_id"], "actual": labels[truth[j]], "top1": labels[raw_pred[j]],
         "decision": labels[decision[j]], "confidence": scores[j].max(),
         "correct": int(truth[j] == decision[j])}
        for j, i in enumerate(indices)
    ])

    before, raw_bins = confidence_quality(truth, raw, "raw")
    after, scaled_bins = confidence_quality(truth, scores, "calibrated")
    write_table(OUTPUT_DIR / f"{name}_confidence_bins.csv", raw_bins + scaled_bins)
    idle = truth == null_id
    visible = decision != null_id
    report = {
        "argmax": metrics(truth, raw_pred),
        "threshold": metrics(truth, decision),
        "confidence_before": before, "confidence_after": after,
        "signer_count": len({rows[i]["group"] for i in indices}),
        "word_display_rate": float(visible.mean()),
        "displayed_word_accuracy": float((decision[visible] == truth[visible]).mean()) if visible.any() else None,
        "false_activation_rate": float(visible[idle].mean()) if idle.any() else None,
    }
    return report, truth, raw_pred


user_test_indices = np.array([i for i in split_indices["test"] if rows[i]["storage"] == "user"], dtype=np.int64)
extra_test_indices = np.array([i for i in split_indices["test"] if rows[i]["storage"] == "extra"], dtype=np.int64)
report, user_truth, user_predictions = evaluate_test("tsl51_user", user_test_indices)
results = {"tsl51_user": report}
if len(extra_test_indices):
    results["project_extra"], _, _ = evaluate_test("project_extra", extra_test_indices)

summary = {"selected_feature_mode": selected_mode, "best_epoch": winner["best_epoch"],
           "confidence_threshold": threshold, "temperature": temperature,
           "split_sizes": {key: len(indices) for key, indices in split_indices.items()},
           "test": results}
print(json.dumps(summary, indent=2))

In [ ]:
def compare_previous_bundle(path):
    if path.is_dir():
        info = json.loads((path / "metadata.json").read_text(encoding="utf-8"))
        previous = torch.jit.load(str(path / "model.torchscript.pt"), map_location="cpu").eval()
    else:
        with ZipFile(path) as bundle:
            metadata_files = [name for name in bundle.namelist() if PurePosixPath(name).name == "metadata.json"]
            if len(metadata_files) != 1:
                raise ValueError("Expected one metadata.json in the baseline ZIP")
            meta = PurePosixPath(metadata_files[0])
            info = json.loads(bundle.read(str(meta)))
            previous = torch.jit.load(
                io.BytesIO(bundle.read(str(meta.with_name("model.torchscript.pt")))), map_location="cpu",
            ).eval()

    previous_labels = [item["sign_id"] for item in info["classes"]]
    if (info["input_features"] != preprocess.INPUT_FEATURES
            or info["coordinate_columns"] != preprocess.COORDINATE_COLUMNS
            or set(previous_labels) != {labels[i] for i in user_truth}):
        raise ValueError("Use the previous 52-class TSL-51 project bundle")
    mapping = np.array([label_index[label] for label in previous_labels])
    previous_prediction = []
    with torch.inference_mode():
        for start in tqdm(range(0, len(user_test_indices), BATCH_SIZE), desc="Previous model"):
            sequences = []
            for i in user_test_indices[start:start + BATCH_SIZE]:
                with (TSL51_DIR / rows[i]["landmark_path"]).open(encoding="utf-8-sig", newline="") as stream:
                    sequences.append(preprocess.read_landmark_csv(stream, info["sequence_frames"]))
            result = previous(torch.from_numpy(np.stack(sequences))).argmax(dim=1).numpy()
            previous_prediction.extend(mapping[result])
    previous_prediction = np.array(previous_prediction)
    old_correct = previous_prediction == user_truth
    new_correct = user_predictions == user_truth
    return {
        "previous_argmax": metrics(user_truth, previous_prediction),
        "new_argmax": metrics(user_truth, user_predictions),
        "accuracy_delta_percentage_points": 100 * float(new_correct.mean() - old_correct.mean()),
        "new_only_correct": int((new_correct & ~old_correct).sum()),
        "previous_only_correct": int((old_correct & ~new_correct).sum()),
    }


if BASELINE_BUNDLE is not None:
    summary["matched_baseline_comparison"] = compare_previous_bundle(Path(BASELINE_BUNDLE))
    print(json.dumps(summary["matched_baseline_comparison"], indent=2))

## 5. ส่งออกโมเดลและตรวจผล

ไฟล์ `model.torchscript.pt` รวม feature engineering ของโหมดที่ชนะและ temperature scaling ไว้ด้วย อินพุตภายนอกยังเป็น 216 ค่าต่อเฟรมจาก `tsl_preprocess.py` ใช้ `sequence_frames` จาก `metadata.json` เมื่อแตก ZIP แล้ว สามารถเปิดเดโมเดิมด้วย `app.py --model-dir <โฟลเดอร์ที่แตก ZIP>` หรือเรียก `from tsl_inference import SignPredictor` แล้วใช้ `SignPredictor(model_dir).predict_csv("clip.csv")` กับคลิปหนึ่งท่า

ใน `ablation.csv` ดูคอลัมน์ `user_calib_macro_f1` ควบคู่ accuracy และจำนวนพารามิเตอร์ ส่วน `metrics.json` จะแสดงผลบน test หลังล็อกการเลือกทั้งหมด เวลา inference ที่วัดด้านล่างคือเวลา forward pass บน CPU ต่อคลิปที่แปลง landmark แล้ว

In [ ]:
%%writefile /content/tsl_inference.py
import json
from pathlib import Path

import torch

from tsl_preprocess import landmarks_to_tensor, read_landmark_csv


class SignPredictor:
    def __init__(self, model_dir):
        model_dir = Path(model_dir)
        self.metadata = json.loads((model_dir / "metadata.json").read_text(encoding="utf-8"))
        self.model = torch.jit.load(str(model_dir / "model.torchscript.pt"), map_location="cpu").eval()

    @torch.inference_mode()
    def _predict(self, sequence):
        probabilities = self.model(torch.from_numpy(sequence).unsqueeze(0)).softmax(dim=1)[0]
        confidence, index = probabilities.max(dim=0)
        index, confidence = index.item(), confidence.item()
        labels = self.metadata["classes"]
        accepted = confidence >= self.metadata["confidence_threshold"]
        chosen = labels[index] if accepted else labels[self.metadata["null_index"]]
        return {
            "sign_id": chosen["sign_id"], "thai": chosen["thai"],
            "raw_sign_id": labels[index]["sign_id"], "raw_confidence": confidence,
            "rejected_low_confidence": not accepted,
            "probabilities": probabilities.tolist(),
        }

    def predict_csv(self, path):
        with Path(path).open(encoding="utf-8-sig", newline="") as stream:
            return self._predict(read_landmark_csv(stream, self.metadata["sequence_frames"]))

    def predict_landmarks(self, coordinates, time_ms=None):
        return self._predict(landmarks_to_tensor(coordinates, self.metadata["sequence_frames"], time_ms))

In [ ]:
import copy
from zipfile import ZIP_DEFLATED


class CalibratedLSTM(nn.Module):
    def __init__(self, classifier, temperature):
        super().__init__()
        self.classifier = classifier
        self.register_buffer("temperature", torch.tensor(temperature, dtype=torch.float32))

    def forward(self, x):
        return self.classifier(x) / self.temperature


metadata = {
    "dataset": REPO,
    "revision": REVISION if DOWNLOAD_TSL51 != "local" else "local",
    "dataset_license": "CC BY-NC-SA 4.0. See the dataset card for third-party terms.",
    "use_web_scraped": INCLUDE_WEB_SCRAPED,
    "use_project_extra": EXTRA_MANIFEST is not None,
    "architecture": "single-layer unidirectional LSTM",
    "feature_mode": selected_mode,
    "lstm_input_features": model.lstm.input_size,
    "sequence_frames": FRAMES,
    "input_features": preprocess.INPUT_FEATURES,
    "coordinate_columns": preprocess.COORDINATE_COLUMNS,
    "output_type": "temperature-scaled class logits",
    "classes": [{"index": i, "sign_id": label, "thai": thai_names[label]} for i, label in enumerate(labels)],
    "confidence_threshold": threshold,
    "temperature": temperature,
    "null_index": null_id,
    "parameters": winner["parameters"],
    "train_config": {
        "seed": SEED, "epochs": EPOCHS, "batch_size": BATCH_SIZE, "lr": LEARNING_RATE,
        "hidden_size": HIDDEN_SIZE, "dropout": DROPOUT, "patience": PATIENCE,
        "expert_val_fraction": VAL_FRACTION, "max_aug_per_original": MAX_AUG_PER_ORIGINAL,
    },
    "versions": {"torch": str(torch.__version__), "numpy": np.__version__},
}
(OUTPUT_DIR / "metadata.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
shutil.copyfile(checkpoint, OUTPUT_DIR / "best_weights.pt")
shutil.copyfile(preprocess.__file__, OUTPUT_DIR / "tsl_preprocess.py")
shutil.copyfile("/content/tsl_inference.py", OUTPUT_DIR / "tsl_inference.py")

model_file = OUTPUT_DIR / "model.torchscript.pt"
cpu_model = CalibratedLSTM(copy.deepcopy(model).cpu().eval(), temperature).eval()
example = torch.from_numpy(np.asarray(features[user_test_indices[:2]]))
with torch.inference_mode():
    traced = torch.jit.trace(cpu_model, example[:1])
    traced.save(str(model_file))
    reloaded = torch.jit.load(str(model_file)).eval()
    torch.testing.assert_close(reloaded(example), cpu_model(example))
    for _ in range(5):
        reloaded(example[:1])
    elapsed_ms = []
    for _ in range(30):
        start = time.perf_counter()
        reloaded(example[:1])
        elapsed_ms.append((time.perf_counter() - start) * 1000)

summary["export"] = {
    "model_bytes": model_file.stat().st_size,
    "cpu_model_forward_median_ms": float(np.median(elapsed_ms)),
    "cpu_threads": torch.get_num_threads(),
}
(OUTPUT_DIR / "metrics.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

import tsl_inference
tsl_inference = importlib.reload(tsl_inference)
predictor = tsl_inference.SignPredictor(OUTPUT_DIR)
example_row = rows[user_test_indices[0]]
prediction = predictor.predict_csv(TSL51_DIR / example_row["landmark_path"])
assert len(prediction["probabilities"]) == len(labels)
assert math.isclose(sum(prediction["probabilities"]), 1.0, abs_tol=1e-5)
print("ตัวอย่าง | คำจริง:", example_row["sign_id"], "| คำทำนาย:", prediction["sign_id"])
print(summary["export"])

In [ ]:
import matplotlib.pyplot as plt

fig, (learning_ax, confidence_ax) = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
for mode in FEATURE_MODES:
    with (OUTPUT_DIR / f"{mode}_history.csv").open(encoding="utf-8-sig", newline="") as stream:
        records = list(csv.DictReader(stream))
    learning_ax.plot([int(r["epoch"]) for r in records], [float(r["val_macro_f1"]) for r in records],
                     marker=".", label=mode)
learning_ax.set(xlabel="Epoch", ylabel="Macro-F1", title="Expert validation", ylim=(0, 1.02))
learning_ax.legend(loc="lower right")

with (OUTPUT_DIR / "tsl51_user_confidence_bins.csv").open(encoding="utf-8-sig", newline="") as stream:
    bins = list(csv.DictReader(stream))
confidence_ax.plot([0, 1], [0, 1], color="gray", linestyle="--", label="Perfect agreement")
for name in ("raw", "calibrated"):
    subset = [row for row in bins if row["score"] == name]
    confidence_ax.plot([float(r["confidence"]) for r in subset], [float(r["accuracy"]) for r in subset],
                       marker="o", label=name)
confidence_ax.set(xlabel="Mean confidence", ylabel="Observed accuracy", title="TSL-51 user test",
                  xlim=(0, 1.02), ylim=(0, 1.02))
confidence_ax.legend(loc="lower right")
fig.savefig(OUTPUT_DIR / "results.png", dpi=160)
plt.show()

In [ ]:
archive_files = [
    "model.torchscript.pt", "best_weights.pt", "metadata.json", "metrics.json",
    "tsl_inference.py", "tsl_preprocess.py", "splits.csv", "ablation.csv", "results.png",
    *[f"{mode}_history.csv" for mode in FEATURE_MODES],
    *[f"{name}_{kind}.csv" for name in results for kind in
      ("per_class", "confusion", "predictions", "confidence_bins")],
]
bundle_path = OUTPUT_DIR / f"{RUN_NAME}_bundle.zip"
with ZipFile(bundle_path.with_suffix(".partial"), "w", compression=ZIP_DEFLATED) as bundle:
    for name in archive_files:
        bundle.write(OUTPUT_DIR / name, arcname=name)
os.replace(bundle_path.with_suffix(".partial"), bundle_path)
print(f"ZIP พร้อมดาวน์โหลด: {bundle_path}")

In [ ]:
# ตั้ง True แล้วรันเซลล์นี้เพื่อดาวน์โหลด ZIP จาก Colab
DOWNLOAD_BUNDLE = False
if DOWNLOAD_BUNDLE:
    from google.colab import files
    files.download(str(bundle_path))

## 6. ดาวน์โหลดชุดเสริมจากรายงาน

เซลล์นี้เลือกเปิดทีละแหล่ง ใช้เก็บข้อมูลสำหรับงานต่อยอด โดยการเทรนด้านบนอ่าน TSL‑51 และชุดที่ระบุใน `EXTRA_MANIFEST` เท่านั้น ชุด ThaiSignVis มีวิดีโอจำนวนมาก จึงควรเลือกไฟล์หรือโฟลเดอร์จากรายการ Kaggle ในการทดลองครั้งแรก หาก Kaggle ขอสิทธิ์ ให้เข้าเว็บไซต์ยอมรับเงื่อนไขและเพิ่ม `KAGGLE_API_TOKEN` ผ่านเมนู Secrets ของ Colab [วิธีใช้ Kaggle Hub](https://github.com/Kaggle/kagglehub)

สำหรับ One‑Stage‑TFS ให้กด **Download All** ที่ [Mendeley Data](https://data.mendeley.com/datasets/rknd3wbz42/1) แล้วเก็บ ZIP ไว้ใน Drive ส่วน TSL‑ONE‑S แบบ MP4 ให้ขอผ่าน [ฟอร์มโครงการ](https://tsl-one.github.io/) DTFS, JAIST TFS และ Thai Sign Digits ให้ติดต่อผู้วิจัยเพื่อขอไฟล์

In [ ]:
DOWNLOAD_TSL_ONE_KEYPOINTS = False
DOWNLOAD_ATFS_REPOSITORY = False
DOWNLOAD_THAISIGNVIS = False
THAISIGNVIS_PATH = None   # ระบุ path ตามหน้า Kaggle หรือใช้ None เพื่อดาวน์โหลดทั้งชุด

def download_archive(url, destination, signature):
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.is_file():
        with destination.open("rb") as stream:
            if stream.read(len(signature)) == signature:
                return destination
    partial = destination.with_suffix(".part")
    subprocess.check_call([
        "curl", "--fail", "--location", "--retry", "4", "--retry-all-errors",
        "--connect-timeout", "30", "--output", str(partial), url,
    ])
    with partial.open("rb") as stream:
        if stream.read(len(signature)) != signature:
            raise ValueError(f"Unexpected archive format from {url}")
    os.replace(partial, destination)
    return destination


if DOWNLOAD_TSL_ONE_KEYPOINTS:
    print(download_archive(
        "https://raw.githubusercontent.com/TSL-ONE/TSL-ONE.github.io/"
        "62ff4924873fbb20b8691bc36591dfa72c0b4f9a/data/TSL-ONE-Pose.7z",
        WORK_DIR / "public_datasets/TSL-ONE-Pose.7z",
        b"7z\xbc\xaf\x27\x1c",
    ))

if DOWNLOAD_ATFS_REPOSITORY:
    print(download_archive(
        "https://codeload.github.com/beebrain/ATFS_Thai_Finger_Spelling/zip/refs/heads/master",
        WORK_DIR / "public_datasets/ATFS_repository.zip", b"PK",
    ))

if DOWNLOAD_THAISIGNVIS:
    if importlib.util.find_spec("kagglehub") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kagglehub"])
    import kagglehub

    print(kagglehub.dataset_download(
        "thanawuttimpitak/thaisignvis", path=THAISIGNVIS_PATH,
        output_dir=str(WORK_DIR / "public_datasets/ThaiSignVis"),
    ))